# Kest Lineage Tracking & Integrity Verification

This notebook demonstrates how to use Kest to track data lineage, visualize the resulting DAG as an ASCII tree, and finally prove that tampering with historical outputs immediately invalidates the cryptographic signature.

In [ ]:
import copy

from kest import config, render_passport, verified
from kest.core.crypto import generate_keypair, sign_passport
from kest.core.policy import LocalOpaEngine

# 1. Setup Kest Environment & Keys
priv_key, pub_key = generate_keypair()
# 2. Configure Verification Key to actively reject bad actors at ingress
config.verification_key = pub_key
# 3. Setup OPA engine for policy enforcement
config.policy_engine = LocalOpaEngine()
policy = """
package kest.policy
default allow = false
allow {
    not unsafe_mix
}
unsafe_mix {
    has_pii
    not has_stripped
}
has_pii { input.taints[_] == "pii_data" }
has_stripped { input.taints[_] == "pii_stripped" }
"""
config.policy_engine.add_policy("access", policy)


# 4. Define Tainted Functions
@verified(added_taint=["pii_data"])
def ingest_pii(data: dict):
    return data


@verified(added_taint=["internet_data"])
def fetch_internet_data(query: str):
    return {"source": "internet", "query": query}


@verified(added_taint=["pii_stripped"])
def clean_pii(data: dict):
    clean = data.copy()
    if "ssn" in clean:
        clean["ssn"] = "***-**-****"
    return clean


@verified(enforce_rules=["data.kest.policy.allow"])
def merge_results(a, b):
    return {"merged": True, "a": a, "b": b}

### Demonstrate Policy Failure

Our OPA policy enforces that `merge_results` cannot accept `pii_data` unless it also has the `pii_stripped` taint. Let's try merging unstripped data.

In [ ]:
try:
    raw_pii = ingest_pii({"user": "Bob", "ssn": "999-99-999"})
    web_data = fetch_internet_data("news")
    # This should trigger a PermissionError because the PII hasn't been stripped
    merge_results(raw_pii, web_data)
    print("❌ FAIL: Executed when it shouldn't have!")
except PermissionError as e:
    print(f"✅ POLICY ENFORCED! Execution blocked: {e}")

### Execute the Pipeline

We will run a flow that ingests PII, cleans it, fetches internet data, and then merges them.

In [2]:
# Ingest and process
raw_pii = ingest_pii({"user": "Alice", "ssn": "123-45-678"})
cleaned_pii = clean_pii(raw_pii)

# Fetch internet data
web_data = fetch_internet_data("weather")

# Merge everything
final_result = merge_results(cleaned_pii, web_data)

print("Final Data:", final_result.data)
print("Lineage Nodes:", len(final_result.passport.history))

Final Data: {'merged': True, 'a': {'user': 'Alice', 'ssn': '***-**-****'}, 'b': {'source': 'internet', 'query': 'weather'}}
Lineage Nodes: 4


### Visualize the DAG (ASCII Tree)

We use `render_passport` to print the recursive graph history, showing node merges.

In [3]:
print("---- DAG LINEAGE ----")
ascii_tree = render_passport(final_result.passport.model_dump())

# Sign the final passport so we can simulate tampering
signed_passport = sign_passport(priv_key, final_result.passport)
print(f"\nPassport Signed. Signature: {signed_passport.signature[:50]}...")

---- DAG LINEAGE ----
Kest Lineage DAG (Leaf-to-Root):
└── merge_results [Taints: pii_data, internet_data, pii_stripped]
    ├── clean_pii [Taints: pii_data, pii_stripped]
    │   └── ingest_pii [Taints: pii_data]
    └── fetch_internet_data [Taints: internet_data]


Passport Signed. Signature: Gdq6nFyhHvWryFPaAF9VzRPhTE2VgUoa6hy2/YiKP169sJr1Ob...


### Simulate Malicious Tampering & Active Rejection

What happens if an attacker intercepts the result and historically alters the passport to remove the `pii_data` taint from the ingest node? \n
If we configure Kest with a `verification_key`, any `@verified` boundary function will subsequently enforce the ED25519 signature before allowing execution.


In [ ]:
from kest.core.models import KestData


# 1. Define a final egress function
@verified()
def final_export(payload):
    return "Exported successfully to downstream system!"


# 2. Maliciously tamper with a historical node
print("Simulating attacker dropping 'pii_data' taint from historical origin...")
tampered_passport = copy.deepcopy(signed_passport)

for entry_id, entry in tampered_passport.history.items():
    if entry.node_id == "ingest_pii":
        entry.accumulated_taint = []  # Wipe the historical taint label
        entry.added_taint = []

# Wrap tampered payload back up
tampered_data = KestData(data=final_result.data, passport=tampered_passport)

# 3. Attempt to process the tampered data in a secure boundary
try:
    print("\nAttempting to run final_export with tampered data...")
    final_export(tampered_data)
    print("\n❌ FAIL: Executed when it shouldn't have!")
except Exception as e:
    print(f"\n✅ INGRESS REJECTED. Tampering Detected! Error:\n{e}")

Simulating attacker dropping 'pii_data' taint from historical origin...

Attempting to run final_export with tampered data...

✅ INGRESS REJECTED. Tampering Detected! Error:
Invalid DAG Signature: Passport has been maliciously altered.
